# JRA-3Q 海面更正気圧（気圧配置・日本域）一括ダウンロード（Colab版）

手元のPC/ネットワークから `github.com` や GDEX（データ配布元）へのHTTPS通信がブロックされる環境向けに、Google Colab上でダウンロードするノートブックです。Colab（Googleのサーバー）から直接ダウンロードするので、手元の回線・セキュリティソフトの制限は関係なくなります。

対象は `scripts/list_damage_gaps.py --kind pressure` が挙げる台風
（TYDB被害データはあるのに気圧配置がまだ無いもの）のtrack期間の月です。
`fetch_tydb_damage.py --all` でTYDBカバレッジを広げた直後は数百ヶ月分
になることがあります（2026年08月時点で386ヶ月）。既に取得済みの月が
あれば自動的にスキップされます。

## 方式：Google Drive上のフォルダに直接保存
GDEXの**静的ファイルサーバー**から月ごとの全球ファイル（約84MB）を普通にダウンロードし、Colab上で日本周辺（既定: 北緯15〜50度、東経115〜155度）だけを切り出して約4.8MBで保存、元の全球ファイルはすぐ削除します。

切り出し済みファイルは**Google Driveの`MyDrive/jra3q_pressure_japan`に
直接保存**します（③でマウント）。既にそこにある月はファイル存在チェック
だけで即座にスキップされるので、ZIPのアップロード/ダウンロードや展開の
待ち時間が無くなります。`notebooks/build_pressure_json_colab.ipynb`も
同じDriveフォルダをデフォルトで読みに行くので、このノートブックで
保存したファイルはそのまま次のノートブックから使えます。

- **通信量**は1ヶ月あたり約84MB（Colab⇔GDEX間なので手元の回線は使いません）
- Colabのディスクは圧迫しません（全球ファイルは1件ずつ使い捨て。切り出し後のファイルはDriveへ直接書き込み）

### なぜ「一見無駄な」全球ダウンロードなのか
サーバー側で切り出してもらう賢い方法を2つ試しましたが、どちらもこのサーバーでは使えませんでした:
- **NCSS**（サーバー側切り出し）… 大半の月がタイムアウト。成功率1割以下
- **OPeNDAP**（必要な範囲だけ読む）… 単発では13.5秒で成功したが、並列で504、逐次でも504、5日ずつに分割しても504。最後はメタデータを開くだけの最小リクエストすら失敗する状態に

一方で**静的ファイルサーバーは一度も失敗していません**（重い処理をせず、ただファイルを配信するだけのため）。通信量と引き換えに確実さを取る構成にしています。

## セッションが切れたら
ファイルはDriveに直接保存されるので、セッションが切れても①からやり
直せば（Driveの再マウントだけで）既に保存済みの月は自動的にスキップ
され、続きから再開されます。

## 使い方
**⑥を先に実行**してから、①→②→③→④の順で実行してください。

## ① リポジトリを取得（初回はクローン、2回目以降は最新化のみ）

In [ ]:
import os

REPO_DIR = '/content/typhoon-wind-rainfall'
BRANCH = 'claude/remaining-tasks-gzid2a'

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch {BRANCH} https://github.com/awg-yk/typhoon-wind-rainfall {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

## ② 必要なライブラリをインストール

In [ ]:
!pip install -q xarray netCDF4

## ③ Google Driveをマウント（保存先）

切り出し済みの`.nc`ファイルの保存先を`MyDrive/jra3q_pressure_japan`に
します。既にこのフォルダに前回分のファイルがあれば、④はファイルの
有無をそこで直接チェックして自動スキップします。フォルダ名を変えたい
場合は下のセルの`LOCAL_DIR`を書き換えてください。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
LOCAL_DIR = '/content/drive/MyDrive/jra3q_pressure_japan'
pathlib.Path(LOCAL_DIR).mkdir(parents=True, exist_ok=True)

n = len(list(pathlib.Path(LOCAL_DIR).glob('*.nc')))
print(f'{LOCAL_DIR} に既存の.ncファイル {n} 個（③はこの分をスキップします）')

## ④ ダウンロード実行（本体）

全球ファイルを1件ずつ取得 → 日本域に切り出し → Google Driveの
`MyDrive/jra3q_pressure_japan`へ保存 → 全球ファイル削除、を（既に
そこにある月はスキップして残りの月だけ）繰り返します。各行に経過
時間と残り時間の目安が出ます。

静的ファイルサーバーは同時アクセスに耐えるので、既定で2並列です。速くしたい場合は `--workers 4` などに上げてみてください（失敗が増えるようなら戻す）。地上気圧も欲しい場合は `--include-surface-pressure`、切り出す範囲を変えたい場合は `--north/--south/--west/--east` を追加してください。

最後まで走っても失敗が残った場合は、このセルをもう一度実行すると、失敗した月だけ再取得を試みます（成功済みのファイルはスキップされます）。

In [ ]:
LOCAL_DIR = LOCAL_DIR if 'LOCAL_DIR' in dir() else '/content/drive/MyDrive/jra3q_pressure_japan'

!cd {REPO_DIR} && python scripts/download_jra3q_pressure.py --out-dir "{LOCAL_DIR}" --codes $(python3 scripts/list_damage_gaps.py --kind pressure)


## ⑤ （任意・バックアップ用）ZIPにまとめてPCへダウンロード

ファイルは既にGoogle Driveに保存済みなので、次の
`build_pressure_json_colab.ipynb`はこのDriveフォルダをそのまま読みに
行けます。PCにも控えを残したい場合だけ、このセルを実行してください。

In [ ]:
import shutil
import pathlib

from google.colab import files as colab_files

files = list(pathlib.Path(LOCAL_DIR).glob('*.nc'))
total_mb = sum(f.stat().st_size for f in files) / 1e6
print(f'{len(files)} files, {total_mb:.1f} MB')

zip_base = '/content/jra3q_pressure_japan'
zip_path = shutil.make_archive(zip_base, 'zip', LOCAL_DIR)
print(f'{zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
colab_files.download(zip_path)

## ⑥ （最初に実行してください）無操作切断を遅らせる

④は時間がかかるため、その間にColabが無操作と判断して切断することがあります。**④より先に**このセルを流しておくと、ブラウザのタブを開いたままにしている間は接続維持の合図を送り続けます（非公式の小技のため過信せず、切れたら①からやり直してください）。

In [ ]:
from IPython.display import Javascript, display

display(Javascript('''
function KeepAlive(){
  console.log("keep-alive ping");
  document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(KeepAlive, 60000);
'''))

## 完了後

取得したファイルはGoogle Driveの`MyDrive/jra3q_pressure_japan`に
保存済みです。`notebooks/build_pressure_json_colab.ipynb`を実行すると
同じフォルダをデフォルトで読みに行くので、そのまま続けて実行して
ください（ZIPのダウンロード・展開は不要です）。